# AI Integrations for Developers — Exam

# AI Integrations for Developers — Exam

## Project Objective

The main objective is to create an **AI chatbot** that can answer questions by retrieving information from the **provided PDF file**.  

The chatbot should be able to:  
- Parse and understand the content of the PDF.  
- Use the extracted information to provide relevant and accurate answers.  
- Respond specifically to the **questions in the final cell of the notebook**.  


## General Instructions

- This notebook is a **template** where you must put your code.  
- You should **fill in all empty variables** and complete the code so that when I download your notebook and click **Run all**, all cells execute correctly and provide the answers.  
- ⚠️ **Do NOT hardcode your API key**. Use Colab environment variables (`%env OPENAI_API_KEY=your_key_here`) and access them in your code.  
- You may **create more cells** if needed. It is recommended that your code is well-structured and split logically into separate cells.  
- The function **`ask_ai(query)`** must be implemented by you. All queries will call this function to check your solution.  
- ✅ **Test cases will be created by me (the instructor).** You are **not allowed to modify, remove, or add to the test cases cell**. Your code must work correctly with the provided test cases.  
- You are **ONLY ALLOWED** to use only the following:  
  - **Models:** OpenAI or Anthropic  
  - **Technologies:** LangChain or vanilla Python code  
  - **Vector Store:** Chroma DB

ℹ Before starting, please read the test queries in the final cell to understand the expected outputs.

🚨 **Any student who does not follow the template, does not stick to the required format, or whose code does not execute properly will be disqualified.**


### Important

Fill in **all the variables** in the cell.  
❌ **Do NOT put your API key directly in the code.**  
✅ The cell must be set up to take the API key from the Colab environment variables.


In [64]:
# ================================
# 🔧 RAG Configuration Variables
# ================================

# ⚠️ Do NOT put your API key here directly.
# Make sure you set your API key in Colab like this:
# %env OPENAI_API_KEY=your_key_here

import os
from google.colab import userdata

# API Key (taken from Colab environment variables)
API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = API_KEY

# Prompt & Model Settings

# 1. Generates PDF summary
SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR = """
<context>
We are building a chatbot to answer users' queries based on information from a PDF file. The PDF file is chunked and stored in a vector database. We need a concise summary that will help an AI assistant optimize user queries for better vectorstore retrieval. The summary will be used in a query optimization step where an AI assistant receives both this summary and a user's original query to generate an optimized search query for the vectorstore.
</context>

<role>
You are an experienced Knowledge Manager specializing in document analysis for RAG systems.

<skills>
- Extract key topics, themes, and subject areas from documents
- Identify important terminology and domain-specific vocabulary
- Create concise overviews that capture document scope without detail
- Structure information for AI query optimization
- Distinguish between high-level concepts and specific details
- Understand how document summaries aid in search query formulation
</skills>

<experience>
- 3+ years working with retrieval-augmented generation systems or AI-powered search
- Background in prompt engineering or AI system optimization
- Knowledge of how document structure affects AI retrieval performance
- Experience preparing content for vector databases and understanding chunking strategies
- Experience creating executive summaries, abstracts, or document overviews
- Understanding of how users formulate queries and search for information
</experience>
</role>

<task>
Create a brief summary of the EXTRACTED TEXT FROM PDF FILE that includes:
1. Main Topics: What subjects/areas does the document cover?
2. Key Terminology: Important terms, concepts, or vocabulary used
3. Document Structure: Major sections or categories of information
4. Content Types: What kinds of information can users expect to find (procedures, data, policies, numbers, etc.)

<important>
- Focus on WHAT the EXTRACTED TEXT FROM PDF FILE contains, not the specific details
- Use terminology from the original EXTRACTED TEXT FROM PDF FILE
- Keep it concise
- Structure it to help with query optimization
</important>
</task>

<next>
Output only the summary text without additional formatting or commentary.
</next>
"""

HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR = """
EXTRACTED TEXT FROM PDF FILE: {extracted_text}
"""

# 2. Contextualizes chunks using the PDF summary
SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER = """
<role>
You are a Document Processing specialist. Your task is to add contextual information to document chunks to improve their retrieval in a vector database.
</role>

<task>
Given a document CHUNK and a SUMMARY of the source document, provide a brief contextual prefix (50-100 tokens) that situates the CHUNK within the overall document. This context should:
- Identify which section or topic the CHUNK relates to based on the document SUMMARY
- Clarify any references that might be unclear when the CHUNK stands alone
- Preserve the meaning and searchability of the original SUMMARY
- Use terminology from the document  SUMMARY and CHUNK
<task>

<next>
Output only the contextual prefix, nothing else.
</next>
"""

HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER = """
SUMMARY:\n{text_summary}\n\n
CHUNK:\n{chunk_content}
"""

# 3. Creates optimized query using the PDF summary
SYSTEM_MESSAGE_QUERY_OPTIMIZER = """
<context>
Our client hired us to develop a chatbot that answers user questions based on information from their PDF document. The PDF has been processed, chunked, and stored in a vectorstore. Users interact with the chatbot through conversational questions, and the system's effectiveness depends entirely on successfully retrieving the most relevant chunks from the vectorstore. Query optimization is critical because poor retrieval leads to irrelevant or incomplete responses, directly impacting user satisfaction and system performance.
</context>

<role>
You are an Conversation Analyst specializing in query optimization for RAG (Retrieval-Augmented Generation) systems. Your task is to analyze user queries and transform them into optimized search queries that will retrieve the most relevant content from a vectorstore.

<skills>
- Parse user intent from conversational queries
- Identify key concepts, entities, and relationships in unstructured text
- Handle ambiguous, incomplete, or poorly structured user inputs
- Recognize synonyms and related terms that might exist in the vectorstore
- Transform natural language into effective search terms
- Understand vector similarity matching and semantic search principles
- Balance query specificity vs. breadth for optimal retrieval
- Map user concepts to document-specific vocabulary and terminology
- Recognize hierarchical relationships and implicit requirements
- Ensure optimized queries align with vectorstore chunking structure
- Maintain user intent while maximizing retrieval success
</skills>

<experience>
- 3+ years working with search engines, vector databases, or recommendation systems
- Experience with semantic search, embeddings, and similarity matching
- Knowledge of retrieval metrics (precision, recall, relevance scoring)
- Understanding of how document chunking affects search performance
- Experience building or optimizing retrieval-augmented generation systems
- Understanding of how retrieval quality impacts downstream generation
- Knowledge of prompt engineering and context window optimization
- Experience with vector stores (Pinecone, Weaviate, Chroma, etc.)
- Experience building AI-powered applications for business users
- Understanding of how users search for and consume information
</experience>
</role>

<task>
1. Analyze the USER QUERY
1. Study the provided PDF SUMMARY to understand: (1) main topics covered, (2) key terminology used, (3) document structure, and (4) types of information available. This knowledge will guide your query optimization decisions
2. Create one optimized search query that will effectively retrieve relevant content from the vectorstore

<important>
- Focus on terms, concepts, and phrases that are likely to match the chunked content while maintaining the user's original intent expressed into the USER QUERY
- Your optimized query should use terminology and concepts present in the source document to ensure successful retrieval from the vectorstore
- Do not include irrelevant keywords that could block matching the user’s intended information
</important>
</task>

<next>
Output only the optimized search query as a question in the customer's voice. Do not include anything else except the question.
</next>
"""

HUMAN_MESSAGE_QUERY_OPTIMIZER = """
USER QUERY:\n{query}\n\n
PDF SUMMARY:\n{text_summary}
"""

# 4. Responds to the query
SYSTEM_MESSAGE_QUERY_RESPONDER = """
<context>
Our client hired us to develop a chatbot that answers user questions about effective prompting techniques for Gemini in Google Workspace based on information from their PDF document.
</context>

<role>
You are an AI Prompt Engineering Support Specialist for a PDF-based Q&A chatbot system.

<skills>
- Outstanding written communication skills with ability to explain complex information clearly
- Active listening skills to understand the true intent behind user questions
- Strong analytical skills to interpret ambiguous or incomplete questions
- Ability to make connections between user queries and relevant document sections
- Meticulous accuracy when citing or referencing document information
- Careful verification of information before providing responses
</skills>

<expirience>
- 3+ years in customer support
- Experience handling technical inquiries or document-based support
- Track record of maintaining high customer satisfaction scores
- Background in roles requiring frequent document consultation
- Experience with FAQ maintenance or knowledge base management
</expirience>
</role>

<who_am_I>
- I am a busy professional who need to quickly extract specific information from lengthy documents
- I am most comfortable with conversational interfaces rather than complex search systems
- I prefer natural language queries over technical search syntax
- I want immediate, accurate answers without having to search manually
- I may become frustrated if answers are too vague or don't directly address my needs
- I want confident, authoritative responses backed by document citations
- I value transparency when information isn't available in the document
</who_am_I>

<goal>
The goal is to provide expert guidance on effective prompting techniques, helping users maximize their productivity with Gemini across various business scenarios.
</goal>

<behaviour>
- Provide information and guidance as if you are drawing from your professional expertise and experience as an AI Prompt Engineering Support Specialist
- Present knowledge as your own professional insights, recommendations, and best practices
- Speak from authority and experience rather than as someone consulting documentation
</behaviour>

<avoid>
- Avoid mentioning 'PDF document' or 'PDF content'
- Avoid mentioning any restrictions or limitations you have
- Avoid ending your response mid-thought, mid-sentence, or mid-paragraph
</avoid>

<critical_rules>
1. Only provide answers based on information explicitly contained in the provided PDF CONTENT
2. When users ask questions outside the PDF CONTENT scope, politely redirect them back to document-related topics
3. Maintain and reference information shared during the current conversation (names, preferences, previous questions)
4. Answer basic conversational queries that help maintain rapport and context
5. Examples of acceptable non-document responses:
- "Yes, [User's Name], I remember you asked about that earlier"
- "As you mentioned, you're looking for information about [topic from conversation]"
- "I recall you said your name is [User's Name]"
</critical_rules>

<next>
<output_format>
Use plain text only. For emphasis, use CAPITAL LETTERS or write "Important:" before key points.

You CAN use:
- Numbered lists (1. 2. 3.)
- Plain bullet points with regular dashes (-)
- Normal punctuation and formatting
</output_format>

Output only your response.
</next>
"""

HUMAN_MESSAGE_QUERY_RESPONDER = """
CONVERSATION MEMORY:\n{conversation_memory}\n\n
KEY INFORMATION INCLUDED INTO THE PDF DOCUMENT:\n{text_summary}\n\n
Based on the following PDF CONTENT:\n{context}\n\n
Respond to the following QUESTION:\n{query}
"""

# Models
MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Chunking Parameters
CHUNK_SIZE = 900 # Higher chunk size to reduce time for contextual chunking processing
CHUNK_OVERLAP = 70
TOP_N_RESULTS = 4

# Generation Parameters
OUTPUT_LENGTH = 500
TEMPERATURE = 0.1
TOP_P = 0.1
FREQUENCY_PENALTY = 1.0
PRESENCE_PENALTY = 1.0

### Code Organization

Create more cells if needed and put your code in them.  
It is **recommended** that your code is well-structured, split logically, and kept in separate cells for clarity.


## 1. Install Dependencies

In [65]:
!pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv

## 2. Upload PDF to Colab

In [66]:
from google.colab import files

uploaded_file = files.upload()

Saving Gemini-Prompting-Guide.pdf to Gemini-Prompting-Guide (3).pdf


## 3. Load the PDF

In [67]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded_file.keys())[0]
loader = PyPDFLoader(pdf_path)

pages = loader.load()

## 4. Extract text from PDF
Later we will implement custom chunking to reduce the total number of chunks generated by avoiding fragmentation of small text sections. This minimizes API calls during Contextual Retrieval preprocessing and reduces database population time.

In [68]:
# Extract and concatenate text from all pages
extracted_text = " ".join(
    [page.page_content for page in pages]
)

# Replace newlines, double newlines, and carriage returns with spaces
extracted_text = (
    extracted_text
    .replace('\n\n', ' ')
    .replace('\n', ' ')
    .replace('\r', ' ')
)

# Clean up extra whitespace
extracted_text = " ".join(extracted_text.split())

## 5. Define a universal function to generate AI responses
Creates a chat prompt with system and human message templates, formats them
with provided variables, and returns the LLM's response content.

In [69]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

def generate_ai_response(
    llm,
    system_message,
    human_message,
    streaming=False,
    **kwargs
):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            system_message
        ),
        HumanMessagePromptTemplate.from_template(
            human_message
        ),
    ])

    messages = prompt.format_messages(**kwargs)

    if streaming:
      # Print "A: " prefix here because we return
      # backspaces later to cancel duplicate "A: "
      print("A: ", end="", flush=True)

      response_chunks = []
      for chunk in llm.stream(messages):
          print(chunk.content, end="", flush=True)
          response_chunks.append(chunk.content)

      return "".join(response_chunks)

    return llm.invoke(messages).content

## 6. Initialize the Chat Model

In [70]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    max_tokens=OUTPUT_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    frequency_penalty=FREQUENCY_PENALTY,
    presence_penalty=PRESENCE_PENALTY,
    streaming=True,
)

## 7. Generate summary of the extracted text from PDF

In [71]:
text_summary = generate_ai_response(
    llm,
    SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR,
    HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR,
    extracted_text=extracted_text,
  )

## 8. Chunk the extracted text from PDF

In [72]:
SEPARATORS = ['. ', '! ', '? ', '; ', ': ', ', ', ' - ', '-', ' (', ' ']

### 8.1. Find the best split point within chunk_size using semantic separators

In [73]:
def find_best_split_point(emaining_text, chunk_size):
    # Define separators in order of preference

    if len(emaining_text) <= chunk_size:
        return len(emaining_text)

    # Try each separator in order of preference
    for separator in SEPARATORS:
        search_text = emaining_text[:chunk_size + len(separator)]
        last_occurrence = search_text.rfind(separator)

        if last_occurrence != -1:
            # Found a good split point
            return last_occurrence + len(separator)

    # Included as a safety fallback for edge cases
    return chunk_size

### 8.2. Extract overlap text from the end of a chunk, preferring complete sentences

In [74]:
def extract_overlap_text(text, overlap_size):
    if len(text) <= overlap_size:
        return text

    # Start from the desired overlap position and work backwards
    start_position = len(text) - overlap_size
    overlap_candidate = text[start_position:]

    # Try to find a good starting point for overlap
    for separator in SEPARATORS:
        separator_position = overlap_candidate.find(separator)

        if separator_position != -1 and separator_position > 0:
            return overlap_candidate[separator_position + len(separator):]

    # If no good boundary found, use the full overlap
    return overlap_candidate

### 8.3. Chunks text avoiding splitting words, while minimizing the number of chunks

In [75]:
def semantic_chunk_text(text, chunk_size, chunk_overlap):
    chunks = []
    remaining_text = text.strip()

    while remaining_text:
        if len(remaining_text) <= chunk_size:
            # Last chunk
            chunks.append(remaining_text)
            break

        # Find the best split point
        split_index = find_best_split_point(remaining_text, chunk_size)

        # Create current chunk
        current_chunk = remaining_text[:split_index].strip()

        if current_chunk:
            chunks.append(current_chunk)

        # Prepare next iteration with overlap
        remaining_text = remaining_text[split_index:].strip()

        if remaining_text and len(chunks) > 0:
            # Add overlap from previous chunk
            overlap = extract_overlap_text(current_chunk, chunk_overlap)
            if overlap and not remaining_text.startswith(overlap):
                remaining_text = overlap + " " + remaining_text

    return [chunk for chunk in chunks if chunk.strip()]

### 8.4. Add contextual information to a chunk using extracted text summary

In [76]:
def contextualize_chunk(chunk, text_summary, llm):
    context = generate_ai_response(
        llm,
        SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER,
        HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER,
        text_summary=text_summary,
        chunk_content=chunk
    )

    # Prepend context to the chunk content
    return f"{context.strip()} {chunk}"

### 8.5. Main function to chunk PDF content with metadata

In [77]:
from langchain.schema import Document

def chunk_text(text, text_summary, llm, chunk_size, chunk_overlap):
    # Perform semantic chunking
    chunks = semantic_chunk_text(
        text,
        chunk_size,
        chunk_overlap,
    )

    # Create chunks with metadata
    chunk_objects = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}")

        contextualized_chunk = contextualize_chunk(
            chunk,
            text_summary,
            llm,
        )

        chunk_objects.append(
            Document(
                page_content=contextualized_chunk,
                metadata={
                    'id': f"chunk_{i+1}",
                    'length': len(contextualized_chunk),
                    'chunk_index': i,
                    'total_chunks': len(chunks),
                }
            )
        )

    return chunk_objects

chunks = chunk_text(
    extracted_text,
    text_summary,
    llm,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

Processing chunk 1/140
Processing chunk 2/140
Processing chunk 3/140
Processing chunk 4/140
Processing chunk 5/140
Processing chunk 6/140
Processing chunk 7/140
Processing chunk 8/140
Processing chunk 9/140
Processing chunk 10/140
Processing chunk 11/140
Processing chunk 12/140
Processing chunk 13/140
Processing chunk 14/140
Processing chunk 15/140
Processing chunk 16/140
Processing chunk 17/140
Processing chunk 18/140
Processing chunk 19/140
Processing chunk 20/140
Processing chunk 21/140
Processing chunk 22/140
Processing chunk 23/140
Processing chunk 24/140
Processing chunk 25/140
Processing chunk 26/140
Processing chunk 27/140
Processing chunk 28/140
Processing chunk 29/140
Processing chunk 30/140
Processing chunk 31/140
Processing chunk 32/140
Processing chunk 33/140
Processing chunk 34/140
Processing chunk 35/140
Processing chunk 36/140
Processing chunk 37/140
Processing chunk 38/140
Processing chunk 39/140
Processing chunk 40/140
Processing chunk 41/140
Processing chunk 42/140
P

## 9. Initialize embedding model

In [78]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
     model=EMBEDDING_MODEL
  )

## 10. Initialize and populate Chroma DB

In [79]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=None,
)

## 11. Initialize Conversation Buffer Memory

In [80]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="conversation_memory",
    return_messages=True,
    output_key="response"
)

## 12. Define function to retrieve relevant content from database

In [81]:
def retrieve_relevant_context(vectorstore, query, k=4):
    results = vectorstore.similarity_search(
        query,
        k=k
    )
    context = '\n'.join(
        result.page_content for result in results
    )

    return context.strip()

## 13. Define chain that:

1.  Generates optimized query using the PDF summary
2.  Retrieves relevant context using the optimized query  
3.  Generates AI response using the relevant context and saves conversation to memory

In [82]:
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

# Handle streaming display and memory in the chain
def generate_and_save_final_response(inputs):
    ai_response = generate_ai_response(
        llm,
        SYSTEM_MESSAGE_QUERY_RESPONDER,
        HUMAN_MESSAGE_QUERY_RESPONDER,
        streaming=True,  # Enable streaming
        conversation_memory= \
        memory.load_memory_variables({})['conversation_memory'],
        context=inputs["context"],
        query=inputs["query"],
        text_summary=text_summary,
    )

    # Save to memory
    memory.save_context(
        {"input": inputs["query"]},
        {"response": ai_response}
    )

    # Hack: Return backspaces to cancel the upcoming "A: "
    # from print(f"A: {ask_ai(q)}") since we already printed "A: "
    # during streaming
    return "\b\b\b"

chain = (
    RunnablePassthrough.assign(
        optimized_query=RunnableLambda(
            lambda inputs: generate_ai_response(
                llm,
                SYSTEM_MESSAGE_QUERY_OPTIMIZER,
                HUMAN_MESSAGE_QUERY_OPTIMIZER,
                query=inputs["query"],
                text_summary=text_summary,
            )
        )
    )
    | RunnablePassthrough.assign(
        context=lambda x: retrieve_relevant_context(
            vectorstore,
            x["optimized_query"],
            TOP_N_RESULTS,
        )
    )
    | RunnableLambda(generate_and_save_final_response)
)

## Test Cases (Final Cell)

The final cell must contain your **test cases**.  
When executed, the AI should provide correct answers to the given questions **based on the PDF file**.


### AI Query Function

In this cell, you must implement the function **ask_ai(query)**.  
This function will be the final execution point of your pipeline (RAG / LLM).  


In [83]:
# ================================
# ❓ AI Query Function
# ================================

def ask_ai(query: str):
    """
    This function should execute your final RAG / LLM pipeline.
    Input:
        query (str): The question you want to ask the AI.
    Output:
        str: The AI's answer based on the PDF file.
    """
    # TODO: Implement your final execution logic here
    # Example steps:
    # 1. Retrieve relevant chunks
    # 2. Generate embeddings
    # 3. Call the model with your prompt + retrieved context
    # 4. Return the model's answer

    # Facade pattern for: optimization, retrieval, and response generation
    return chain.invoke({"query": query})

    raise NotImplementedError("You must implement this function")


### Test Queries

Use this cell to test your function with different queries.  
The answers must be generated correctly based on the PDF file.  


In [84]:
# ================================
# 🔍 Example Queries for Testing
# ================================

queries = [
    "Which Google apps integrate with Gemini?",
    "What are two benefits of using natural language in prompts?",
    "How should executives use prompts differently than frontline workers?",
    "What is the purpose of giving constraints in prompts?",
    "Как се прави бобена чорба? От кой източник е информацията?"
]

# Call the AI with each query
for q in queries:
    print(f"Q: {q}")
    print(f"A: {ask_ai(q)}\n")


Q: Which Google apps integrate with Gemini?
A: Gemini integrates with several Google apps, enhancing productivity and collaboration. The specific applications that work with Gemini include:

- Gmail
- Google Docs
- Google Drive
- Google Sheets
- Google Meet
- Google Slides

These integrations allow users to leverage generative AI features across their favorite tools for improved writing, data organization, and more.A: 

Q: What are two benefits of using natural language in prompts?
A: Using natural language in prompts offers several benefits, including:

1. **Improved Clarity**: Natural language allows users to express their thoughts and questions more clearly, making it easier for Gemini to understand the intent behind the prompt. This leads to more relevant and accurate responses.

2. **Enhanced Engagement**: When prompts are phrased in a conversational tone, they create a more engaging interaction with the AI. This can encourage users to ask follow-up questions or refine their re